In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu

In [2]:
from google.colab import files

uploaded = files.upload()

Saving 2410.12837v1.pdf to 2410.12837v1 (4).pdf


In [3]:
import fitz

pdf_filename = next(iter(uploaded.keys()))

doc = fitz.open(pdf_filename)

text = ""

for page in doc:
    text += page.get_text() + "\n"

print("Total pages:", len(doc))
print("Total characters:", len(text))
print("\nFirst 2000 characters:\n")
print(text[:2000])

Total pages: 18
Total characters: 53990

First 2000 characters:

A
Comprehensive
Survey
of
Retrieval-Augmented
Generation
(RAG):
Evolution,
Current
Landscape and Future Directions
Shailja Gupta (Carnegie Mellon University, USA)
Rajesh Ranjan (Carnegie Mellon University, USA)
Surya Narayan Singh (BIT Sindri, India)
Abstract
This paper presents a comprehensive study of Retrieval-Augmented Generation (RAG), tracing its
evolution from foundational concepts to the current state of the art. RAG combines retrieval mechanisms
with generative language models to enhance the accuracy of outputs, addressing key limitations of LLMs.
The study explores the basic architecture of RAG, focusing on how retrieval and generation are integrated
to handle knowledge-intensive tasks. A detailed review of the significant technological advancements in
RAG is provided, including key innovations in retrieval-augmented language models and applications
across various domains such as question-answering, summarizatio

In [4]:
!pip install -q langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Total chunks: 73

First chunk:

A
Comprehensive
Survey
of
Retrieval-Augmented
Generation
(RAG):
Evolution,
Current
Landscape and Future Directions
Shailja Gupta (Carnegie Mellon University, USA)
Rajesh Ranjan (Carnegie Mellon University, USA)
Surya Narayan Singh (BIT Sindri, India)
Abstract
This paper presents a comprehensive study of Retrieval-Augmented Generation (RAG), tracing its
evolution from foundational concepts to the current state of the art. RAG combines retrieval mechanisms
with generative language models to enhance the accuracy of outputs, addressing key limitations of LLMs.
The study explores the basic architecture of RAG, focusing on how retrieval and generation are integrated
to handle knowledge-intensive tasks. A detailed review of the significant technological advancements in
RAG is provided, including key innovations in retrieval-augmented language models and applications
across various domains such as question-answering, summarization, and knowledge-based tasks.


In [6]:
!pip install -q sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert text chunks into embeddings
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Number of chunks: 73
Embedding shape: (73, 384)


In [8]:
!pip install -q faiss-cpu

In [9]:
import faiss
import numpy as np

# Convert embeddings to the format FAISS requires
embeddings = np.asarray(embeddings, dtype="float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# Create the search index
index = faiss.IndexFlatIP(embeddings.shape[1])

# Add all paper chunks to the index
index.add(embeddings)

print("FAISS index created successfully!")
print("Total chunks indexed:", index.ntotal)

FAISS index created successfully!
Total chunks indexed: 73


In [10]:
def search_paper(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    for rank, idx in enumerate(indices[0], start=1):
        if idx == -1:
            continue

        print(f"\n--- Result {rank} | Similarity: {scores[0][rank-1]:.3f} ---")
        print(chunks[idx][:1000])


search_paper("What is retrieval-augmented generation and why is it useful?")


--- Result 1 | Similarity: 0.691 ---
A
Comprehensive
Survey
of
Retrieval-Augmented
Generation
(RAG):
Evolution,
Current
Landscape and Future Directions
Shailja Gupta (Carnegie Mellon University, USA)
Rajesh Ranjan (Carnegie Mellon University, USA)
Surya Narayan Singh (BIT Sindri, India)
Abstract
This paper presents a comprehensive study of Retrieval-Augmented Generation (RAG), tracing its
evolution from foundational concepts to the current state of the art. RAG combines retrieval mechanisms
with generative language models to enhance the accuracy of outputs, addressing key limitations of LLMs.
The study explores the basic architecture of RAG, focusing on how retrieval and generation are integrated
to handle knowledge-intensive tasks. A detailed review of the significant technological advancements in
RAG is provided, including key innovations in retrieval-augmented language models and applications
across various domains such as question-answering, summarization, and knowledge-based task

In [11]:
def ask_paper(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    print("QUESTION:", question)
    print("\nRELEVANT INFORMATION FROM THE PAPER:\n")

    for rank, idx in enumerate(indices[0], start=1):
        if idx == -1:
            continue

        print(f"\n--- Source passage {rank} ---")
        print(chunks[idx])

ask_paper("What are the main benefits of retrieval-augmented generation?")

QUESTION: What are the main benefits of retrieval-augmented generation?

RELEVANT INFORMATION FROM THE PAPER:


--- Source passage 1 ---
A
Comprehensive
Survey
of
Retrieval-Augmented
Generation
(RAG):
Evolution,
Current
Landscape and Future Directions
Shailja Gupta (Carnegie Mellon University, USA)
Rajesh Ranjan (Carnegie Mellon University, USA)
Surya Narayan Singh (BIT Sindri, India)
Abstract
This paper presents a comprehensive study of Retrieval-Augmented Generation (RAG), tracing its
evolution from foundational concepts to the current state of the art. RAG combines retrieval mechanisms
with generative language models to enhance the accuracy of outputs, addressing key limitations of LLMs.
The study explores the basic architecture of RAG, focusing on how retrieval and generation are integrated
to handle knowledge-intensive tasks. A detailed review of the significant technological advancements in
RAG is provided, including key innovations in retrieval-augmented language models and appl

In [12]:
!pip install -q -U transformers sentencepiece

In [13]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=150,
    device_map="auto"
)

print("Language model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Language model loaded successfully!


In [14]:
def ask_paper(question, top_k=3):
    # 1. Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # 2. Retrieve relevant chunks
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, top_k)

    valid_indices = [idx for idx in indices[0] if idx != -1]

    context = "\n\n".join(
        chunks[idx] for idx in valid_indices
    )

    # 3. Generate an answer from the retrieved context
    prompt = f"""Answer the question using only the research paper passages.
If the passages do not contain the answer, say that there is not enough
information in the retrieved passages.

Research paper passages:
{context}

Question: {question}
Answer:"""

    result = generator(
        prompt,
        return_full_text=False
    )

    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result[0]["generated_text"])

    print("\nSOURCE PASSAGES:")
    for rank, idx in enumerate(valid_indices, start=1):
        print(f"\nPassage {rank} | Similarity: {scores[0][rank-1]:.3f}")
        print(chunks[idx][:700])
        print("-" * 60)

In [15]:
print("Chunks:", len(chunks))
print("FAISS indexed:", index.ntotal)
print("Generator loaded:", generator is not None)

Chunks: 73
FAISS indexed: 73
Generator loaded: True


In [16]:
result = generator(
    "Question: What is RAG? Answer briefly.",
    max_new_tokens=40,
    do_sample=False,
    return_full_text=False
)

print(result[0]["generated_text"])

 RAG stands for Random Access Generator, a type of random number generator used in computer science and cryptography.

RAGs are designed to generate sequences of bits that appear random but can be generated quickly from


In [25]:
def ask_paper(question, top_k=2):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, top_k)

    valid_indices = [i for i in indices[0] if i >= 0]
    context = "\n\n".join(chunks[i][:700] for i in valid_indices)

    prompt = f"""Use the following research paper text to answer the question.
If the text does not contain the answer, say you cannot determine it.

Research text:
{context}

Question: {question}
Answer briefly:"""

    print("\nSearching the paper...")
    result = generator(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False
    )

    answer = result[0]["generated_text"].strip()

    print("\nANSWER:")
    print(answer if answer else "The model returned an empty answer.")

    print("\nSOURCE PASSAGES:")
    for rank, i in enumerate(valid_indices, start=1):
        print(f"\nPassage {rank} (similarity: {scores[0][rank-1]:.3f})")
        print(chunks[i][:400])

In [18]:
ask_paper("What are the limitations of retrieval-augmented generation?")


Searching the paper...

ANSWER:
The limitations of retrieval-augmented generation include the potential for bias in the retrieved information and the difficulty in integrating diverse sources of knowledge effectively. Additionally, there may be challenges in ensuring that the generated content remains relevant and accurate across different contexts. These issues highlight the ongoing efforts to improve the integration of retrieval and generation techniques in AI systems. To address these limitations, researchers continue to explore new methods and algorithms

SOURCE PASSAGES:

Passage 1 (similarity: 0.628)
A
Comprehensive
Survey
of
Retrieval-Augmented
Generation
(RAG):
Evolution,
Current
Landscape and Future Directions
Shailja Gupta (Carnegie Mellon University, USA)
Rajesh Ranjan (Carnegie Mellon University, USA)
Surya Narayan Singh (BIT Sindri, India)
Abstract
This paper presents a comprehensive study of Retrieval-Augmented Generation (RAG), tracing its
evolution from foundational co

In [19]:
!pip install -q gradio

In [22]:
import gradio as gr
import io
from contextlib import redirect_stdout

def chat_with_paper(question):
    if not question.strip():
        return "Please enter a question."

    output = io.StringIO()

    try:
        with redirect_stdout(output):
            ask_paper(question)

        answer = output.getvalue().strip()
        return answer if answer else "No output was generated."

    except Exception as e:
        return f"Error: {e}"

demo = gr.Interface(
    fn=chat_with_paper,
    inputs=gr.Textbox(
        label="Ask your research paper",
        placeholder="What is RAG?"
    ),
    outputs=gr.Textbox(label="Answer and source passages", lines=18),
    title="Research Paper Intelligence Assistant",
    description="Ask questions and explore retrieved research passages."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0cc3d658cb2af8da94.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2f71debc32d07f256e.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://0cc3d658cb2af8da94.gradio.live
